$$ c_{p, \omega}(a) \coloneq \sum_{e \in a_p} \tau_{\omega, a}(f_e^a) $$

$$\min_{\phi \ge 0, x} \sum_{\omega \in \Omega} \mu_\omega \sum_{a \in A} \phi_{\omega, a} \sum_{p \in N} c_{p, \omega}(a)$$

$$\sum_{\omega \in \Omega} \mu_\omega \sum_{a \in A} c_{p, \omega}(a) \phi_{\omega, a} \le x_{p, s}$$

$$x_{p, v} \le \sum_{\omega \in \Omega} \mu_\omega \sum_{a \in A} c_{e, \omega}(f_e^a + \mathbb{1}_{\{e \notin a_p\}}) \phi_{\omega, a} + x_{p, v'}$$

$$x_{p, t} = 0$$

$$\sum_{a \in A} \phi_{\omega, a} = 1$$

In [1]:
import networkx as nx
import itertools

from bp.world import random_grid_world, Scenario

# NOTE: assuming symmetric players for the purposes of Castiglioni
# NOTE: assuming affine costs (simplifying to beta=1, can consider the step-wise linear func. later)

origin, dest = (1, 0), (1, 3)
n = 15 # keep small bc exponential, solver isn't optimized (doesn't utilize the algorithms to make it poly time to solve)

world = random_grid_world(
    rows=4,
    cols=4,
    demands={
        (origin, dest): n,
    },
    seed=0,
)
G = world.network.graph

world.network.bpr_beta = 1
nominal = Scenario.from_world("nominal", world)

print(f"{world.total_population=}")
print(f"{world.total_demand((1, 0), (1, 3))=}")

world.total_population=15
world.total_demand((1, 0), (1, 3))=15


In [2]:
accident_congestion = 10
target_edge = ((1, 0), (1, 1))

travel_time = dict(nominal.travel_time)
travel_time[target_edge] += accident_congestion

accident = Scenario(
    name="accident",
    travel_time=travel_time,
    discomfort=nominal.discomfort,
    hazard=nominal.hazard,
    cost=nominal.cost,
    emissions=nominal.emissions,
    policing=nominal.policing
)

scenarios = {
    "nominal": (nominal, .8),
    "accident": (accident, .2),
}

assert all(prior >= 0 for _, prior in scenarios.values()), "invalid prior distribution"
assert sum(prior for _, prior in scenarios.values()) == 1, "invalid prior distribution"

In [3]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("castiglioni")
model.setParam("OutputFlag", 0)

V = world.ordered_nodes
A = world.ordered_arcs
I = world.I
N = world.individuals

n = world.total_population
t = world.network.travel_time
c = world.network.capacity
alpha = world.network.bpr_alpha
beta = world.network.bpr_beta

Set parameter Username
Set parameter LicenseID to value 2844113
Academic license - for non-commercial use only - expires 2027-07-13


In [4]:
from collections.abc import Mapping, Sequence

from bp.world import Arc, Node

def edge_path(path: Sequence[Node]) -> Sequence[Arc]:
    return list(itertools.pairwise(path))

# select k shortest paths as possible routes
K = 5
paths = [edge_path(path) for path in itertools.islice(nx.shortest_simple_paths(G, origin, dest, weight="travel_time"), K)]
# TODO: shortest paths are calculated based on the nominal state, which may not be true under the accident state

# NOTE: actually important now, ensure that deviations are only considered for arcs on the admissable paths
active_arcs = set(itertools.chain.from_iterable(paths))

# pre-calculate the induced arc-flows for each action profile
# aligned by index
# action_profiles = list(itertools.product(paths, repeat=n))
action_profiles = list(itertools.combinations_with_replacement(paths, n))
action_profile_flows = []
for action_profile in action_profiles:
    flow = {a: 0 for a in A}
    for path in action_profile:
        for a in path:
            flow[a] += 1
    action_profile_flows.append(flow)

print(f"{len(paths)=}")
print(f"{len(active_arcs)=} ({len(active_arcs) / (len(A) / 2) * 100:.2f}% of arcs)") # / 2 b/c includes backwards edges which are probably irrelevant
print(f"{len(action_profiles)=}")

len(paths)=5
len(active_arcs)=14 (58.33% of arcs)
len(action_profiles)=3876


In [5]:
# reminder: beta=1 is fixed
# tau[omega, a, k] := average cost for k players on arc a under state omega
tau = {}
for scenario_name, (omega, _) in scenarios.items():
    for a in active_arcs:
        for k in range(n + 2): # n + 1 + 1 b/c of deviation indicator variable
            tau[scenario_name, a, k] = omega.travel_time[a] * (1 + alpha * ((k - 1) / c[a]) ** beta)

In [ ]:
# TODO: properly anonymize, may make initialization much faster, which is currently the bottleneck besides profile generation

# decision: phi[omega, a] := \prob[a \mid \theta]
phi = {}
for scenario_name in scenarios:
    for action_profile_idx in range(len(action_profiles)):
        phi[scenario_name, action_profile_idx] = model.addVar(
            vtype=GRB.CONTINUOUS, lb=0, ub=1,
            name=f"phi_{scenario_name}_{action_profile_idx}"
        )

# decision: x[p, v] := minimum expected cost when player p deviates from node v to destination
x = {}
for plr_idx in range(n):
    for v in V:
        x[plr_idx, v] = model.addVar(
            vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY,
            name=f"x_{plr_idx}_{v}"
        )

# constraint: ex-ante persuasiveness
for plr_idx in range(n):
    expected_cost = gp.LinExpr()
    for scenario_name, (omega, mu) in scenarios.items():
        for (action_profile_idx, action_profile), action_profile_flow in zip(enumerate(action_profiles), action_profile_flows):
            cost = sum( # c_{p, \omega}(a)
                tau[scenario_name, a, action_profile_flow[a]]
                for a in action_profile[plr_idx]
            )
            expected_cost += mu * phi[scenario_name, action_profile_idx] * cost
    model.addConstr(
        expected_cost <= x[plr_idx, origin],
        name=f"x_{plr_idx}_{origin}"
    )

# constraint: minimuim cost <= expected cost of deviating
for plr_idx in range(n):
    for a in active_arcs:
        v0, v1 = a
        expected_cost = gp.LinExpr()
        for scenario_name, (omega, mu) in scenarios.items():
            for (action_profile_idx, action_profile), action_profile_flow in zip(enumerate(action_profiles), action_profile_flows):
                deviation_flow = action_profile_flow[a] + (1 if a not in action_profile[plr_idx] else 0)
                cost = tau[scenario_name, a, deviation_flow]
                expected_cost += mu * phi[scenario_name, action_profile_idx] * cost
        model.addConstr(
            x[plr_idx, v0] <= expected_cost + x[plr_idx, v1],
            name=f"x_{plr_idx}_{a}"
        )

# constraint: x[p, t] = 0, zero deviation cost @ dest
for plr_idx in range(n):
    model.addConstr(
        x[plr_idx, dest] == 0,
        name=f"x_{plr_idx}_{dest}"
    )

# constraint: \sum_{a \in A} phi[\omega, a] = 1
for scenario_name in scenarios:
    model.addConstr(
        gp.quicksum(
            phi[scenario_name, action_profile_idx]
            for action_profile_idx in range(len(action_profiles))
        ) == 1,
        name=f"phi_{scenario_name}"
    )

# objective: expected social cost
expected_cost = gp.LinExpr()
for scenario_name, (_, mu) in scenarios.items():
    for (action_profile_idx, action_profile), action_profile_flow in zip(enumerate(action_profiles), action_profile_flows):
        cost = sum(
            sum(
                tau[scenario_name, a, action_profile_flow[a]]
                for a in action_profile[plr_idx]
            )
            for plr_idx in range(n)
        )
        expected_cost += mu * phi[scenario_name, action_profile_idx] * cost
model.setObjective(expected_cost, GRB.MINIMIZE)


In [7]:
model.optimize()
assert model.Status == GRB.OPTIMAL, f"Optimization failed: {model.Status}"

In [8]:
print(f"{model.ObjVal=:.2f}")
print(f"{model.Runtime=:.3f}s")

for scenario_name in scenarios:
    print(f"\nState: {scenario_name}")
    for action_profile_idx, action_profile in enumerate(action_profiles):
        prob = phi[scenario_name, action_profile_idx].X
        if prob > 1e-3:
            print(f"\tProfile {action_profile_idx}: ({prob=:.3f})")
            for plr_idx in range(n):
                print(f"\t\tP{plr_idx} Route: {action_profile[plr_idx]}")

model.ObjVal=434.63
model.Runtime=0.534s

State: nominal
	Profile 141: (prob=1.000)
		P0 Route: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
		P1 Route: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
		P2 Route: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
		P3 Route: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
		P4 Route: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
		P5 Route: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
		P6 Route: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
		P7 Route: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
		P8 Route: [((1, 0), (1, 1)), ((1, 1), (1, 2)), ((1, 2), (1, 3))]
		P9 Route: [((1, 0), (0, 0)), ((0, 0), (0, 1)), ((0, 1), (0, 2)), ((0, 2), (0, 3)), ((0, 3), (1, 3))]
		P10 Route: [((1, 0), (0, 0)), ((0, 0), (0, 1)), ((0, 1), (0, 2)), ((0, 2), (0, 3)), ((0, 3), (1, 3))]
		P11 Route: [((1, 0), (0, 0)), ((0, 0), (0, 1)), ((0, 1), (0, 2)), ((0, 2), (0, 3)), ((0, 3), (1, 3))]
		